# Sistema Especialista: Classificação de Circuitos RLC

**Disciplina:** Inteligencia Artificial — Sistemas Especialistas

**Aluno:** Keven Eduardo Vaz Bilibio

---

Este notebook é uma implementação de um **sistema especialista** simples que classifica o comportamento elétrico de um circuito de corrente alternada (CA) — **resistivo**, **indutivo**, **capacitivo** ou combinados (**RL**, **RC**, **ressonância**) — a partir de observações típicas e conteudos passados nas disciplinas de Laboratorio e Circuitos Eletricos 2, onde atualmente estamos vendo conteudo sobre a defasagem entre tensão e corrente e comparação entre as reatâncias indutiva (X_L) e capacitiva (X_C).


## 1. Domínio

Em um circuito CA, a relação de fase entre a tensão $v(t)$ e a corrente $i(t)$ revela a natureza da carga:

- **Resistor (R):** corrente e tensão estão **em fase** (defasagem = 0°). Fator de potência (FP) = 1.
- **Indutor (L):** a corrente fica **atrasada** 90° em relação à tensão.
- **Capacitor (C):** a corrente fica **adiantada** 90° em relação à tensão.
- **Combinações RL / RC:** produzem uma defasagem **intermediária** (entre 0° e 90°), atrasada ou adiantada conforme o elemento reativo dominante.
- Em uma malha **RLC**, também é possível comparar diretamente as reatâncias:
  $$X_L = 2\pi f L \qquad X_C = \frac{1}{2\pi f C}$$
  Quando $X_L = X_C$ ocorre **ressonância**: as reatâncias se cancelam e a impedância total se reduz à resistência ($Z = R$), fazendo o circuito se comportar como puramente resistivo.

## 2. Base de Conhecimento

| Regra | SE (condição) | ENTÃO (conclusão) | Racional físico |
|:---:|---|---|---|
| **R0a** | `fp = 1` | Deriva `fase = 0°` | FP = cos(fase). Se FP = 1, a defasagem é necessariamente 0° |
| **R0b** | `fp < 1` **e** corrente informada | Deriva `fase = arccos(fp)` (em graus), mantendo o sentido da corrente | Converte o fator de potência em defasagem, alimentando as regras R1–R5 |
| **R0c** | `fp < 1` **e** corrente **não** informada | Alerta: defasagem calculada, mas classificação indefinida | Sem saber se a corrente atrasa ou adianta, não dá pra saber se é RL ou RC |
| **R1** | `fase = 0°` | Circuito puramente **RESISTIVO** | Corrente em fase com a tensão ⇒ FP = 1, sem energia reativa |
| **R2** | `fase = 90°` **e** corrente **atrasada** | Circuito puramente **INDUTIVO** | Indutor atrasa a corrente em 90° |
| **R3** | `fase = 90°` **e** corrente **adiantada** | Circuito puramente **CAPACITIVO** | Capacitor adianta a corrente em 90° |
| **R4** | `0° < fase < 90°` **e** corrente **atrasada** | Circuito **RESISTIVO-INDUTIVO (RL)** | Combinação R+L gera atraso parcial |
| **R5** | `0° < fase < 90°` **e** corrente **adiantada** | Circuito **RESISTIVO-CAPACITIVO (RC)** | Combinação R+C gera adiantamento parcial |
| **R6** | `X_L > X_C` | Predomínio **INDUTIVO** na malha | A reatância indutiva domina a impedância total |
| **R7** | `X_C > X_L` | Predomínio **CAPACITIVO** na malha | A reatância capacitiva domina a impedância total |
| **R8** | `X_L = X_C` (e ambos > 0) | **RESSONÂNCIA** ≈ comportamento **RESISTIVO** puro | As reatâncias se cancelam; $Z = R$ |


## 3. Fatos aceitos pelo sistema

| Atributo | Tipo | Descrição |
|---|---|---|
| `fase` | número (0–90) | Ângulo de defasagem entre tensão e corrente, em graus |
| `corrente` | `"atrasada"` \| `"adiantada"` \| `"em_fase"` | Se a corrente está atrasada, adiantada ou em fase com a tensão |
| `xl` | número | Reatância indutiva (Ω) |
| `xc` | número | Reatância capacitiva (Ω) |
| `fp` | número (0–1) | Fator de potência (FP = cos θ). É convertido internamente em `fase` pelas regras R0a–R0c |


## 4. Implementação

In [ ]:
# Instalação da biblioteca (rode uma vez por ambiente / sessão do Deepnote)
!pip install experta --quiet

import collections, collections.abc
import math

if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping

from experta import KnowledgeEngine, Fact, Rule, MATCH, TEST, NOT, W


In [ ]:
class Circuito(Fact):
    """Fato: características elétricas observadas de um circuito CA."""
    pass


class SistemaEspecialistaCircuitos(KnowledgeEngine):
    """Motor de inferência do sistema especialista de classificação de circuitos RLC."""

    def reset(self, *args, **kwargs):
        super().reset(*args, **kwargs)
        self.resultado = []  # armazena regras e conclusão

    def concluir(self, regra, texto):
        """Registra e exibe uma conclusão."""
        self.resultado.append((regra, texto))
        print(f"[{regra}] {texto}")


    # --- Regras Fator de Potência

    # R0a: SE FP = 1 ENTÃO deriva fase = 0° (reaproveita a regra R1)
    @Rule(Circuito(fp=1))
    def r0a_fp_unitario(self):
        self.concluir("R0a", "FP = 1 equivale a uma defasagem de 0°.")
        self.declare(Circuito(fase=0))

    # R0b: SE FP < 1 E sentido da corrente informado ENTÃO deriva a fase (reaproveita R2-R5)
    @Rule(Circuito(fp=MATCH.fp, corrente=MATCH.corrente), TEST(lambda fp: fp < 1))
    def r0b_fp_com_corrente(self, fp, corrente):
        fase = round(math.degrees(math.acos(fp)), 1)
        self.concluir("R0b", f"FP = {fp} (corrente {corrente}) equivale a uma defasagem de {fase}°.")
        self.declare(Circuito(fase=fase, corrente=corrente))

    # R0c: SE FP < 1 E sentido da corrente NÃO informado ENTÃO avisa que faltam dados
    @Rule(Circuito(fp=MATCH.fp), NOT(Circuito(corrente=W())), TEST(lambda fp: fp < 1))
    def r0c_fp_sem_corrente(self, fp):
        fase = round(math.degrees(math.acos(fp)), 1)
        self.concluir("R0c", f"FP = {fp} equivale a uma defasagem de {fase}°."
            "O sentido da corrente não foi informado, não sendo possível classificar o circuito.")

    # --- Regras fase / corrente

    # R1: SE fase = 0° ENTÃO circuito é puramente RESISTIVO
    @Rule(Circuito(fase=0))
    def r1_resistivo_puro(self):
        self.concluir("R1", "Circuito puramente RESISTIVO (corrente em fase com a tensão, FP = 1).")

    # R2: SE fase = 90° E corrente atrasada ENTÃO circuito é puramente INDUTIVO
    @Rule(Circuito(fase=90, corrente="atrasada"))
    def r2_indutivo_puro(self):
        self.concluir("R2", "Circuito puramente INDUTIVO (corrente atrasada 90° em relação à tensão).")

    # R3: SE fase = 90° E corrente adiantada ENTÃO circuito é puramente CAPACITIVO
    @Rule(Circuito(fase=90, corrente="adiantada"))
    def r3_capacitivo_puro(self):
        self.concluir("R3", "Circuito puramente CAPACITIVO (corrente adiantada 90° em relação à tensão).")

    # R4: SE 0°<fase<90° E corrente atrasada ENTÃO circuito é RESISTIVO-INDUTIVO (RL)
    @Rule(Circuito(fase=MATCH.f, corrente="atrasada"), TEST(lambda f: 0 < f < 90))
    def r4_rl(self, f):
        self.concluir("R4", f"Circuito RESISTIVO-INDUTIVO / RL (defasagem de {f}°, corrente atrasada).")

    # R5: SE 0°<fase<90° E corrente adiantada ENTÃO circuito é RESISTIVO-CAPACITIVO (RC)
    @Rule(Circuito(fase=MATCH.f, corrente="adiantada"), TEST(lambda f: 0 < f < 90))
    def r5_rc(self, f):
        self.concluir("R5", f"Circuito RESISTIVO-CAPACITIVO / RC (defasagem de {f}°, corrente adiantada).")

    # --- Regras comparação de reatâncias

    # R6: SE XL > XC ENTÃO predomínio INDUTIVO
    @Rule(Circuito(xl=MATCH.xl, xc=MATCH.xc), TEST(lambda xl, xc: xl > xc))
    def r6_predominio_indutivo(self, xl, xc):
        self.concluir("R6", f"Predomínio INDUTIVO (X_L={xl}Ω > X_C={xc}Ω).")

    # R7: SE XC > XL ENTÃO predomínio CAPACITIVO
    @Rule(Circuito(xl=MATCH.xl, xc=MATCH.xc), TEST(lambda xl, xc: xc > xl))
    def r7_predominio_capacitivo(self, xl, xc):
        self.concluir("R7", f"Predomínio CAPACITIVO (X_C={xc}Ω > X_L={xl}Ω).")

    # R8: SE XL = XC (>0) ENTÃO RESSONÂNCIA (equivale a RESISTIVO puro)
    @Rule(Circuito(xl=MATCH.xl, xc=MATCH.xc), TEST(lambda xl, xc: xl == xc and xl > 0))
    def r8_ressonancia(self, xl, xc):
        self.concluir("R8", f"Circuito em RESSONÂNCIA (X_L = X_C = {xl}Ω) — equivalente a RESISTIVO puro.")


def diagnosticar(**observacoes):
    """Interface simples: declara os fatos, roda o motor de
    inferência e devolve as conclusões alcançadas."""
    print(f"Fatos declarados: {observacoes}")
    engine = SistemaEspecialistaCircuitos()
    engine.reset()
    engine.declare(Circuito(**observacoes))
    engine.run()
    if not engine.resultado:
        print("Nenhuma regra da base foi disparada para estes fatos "
            "(informações insuficientes ou fora do escopo do sistema).")
    print()
    return engine.resultado

## 5. Testes

In [11]:
# --- Fase e tipo de corrente
diagnosticar(fase=0)                              # esperado: R1 (resistivo puro)
diagnosticar(fase=90, corrente="atrasada")        # esperado: R2 (indutivo puro)
diagnosticar(fase=90, corrente="adiantada")       # esperado: R3 (capacitivo puro)
diagnosticar(fase=45, corrente="atrasada")        # esperado: R4 (RL)
diagnosticar(fase=30, corrente="adiantada")       # esperado: R5 (RC)

# --- Comparação de reatâncias
diagnosticar(xl=50, xc=30)                        # esperado: R6 (predomínio indutivo)
diagnosticar(xl=20, xc=80)                        # esperado: R7 (predomínio capacitivo)
diagnosticar(xl=40, xc=40)                        # esperado: R8 (ressonância)

# --- Fator de potência (FP)
diagnosticar(fp=1)                                # esperado: R0a -> R1 (resistivo puro)
diagnosticar(fp=0.71, corrente="atrasada")        # esperado: R0b -> R4 (RL)
diagnosticar(fp=0.5, corrente="adiantada")        # esperado: R0b -> R5 (RC)
diagnosticar(fp=0, corrente="atrasada")           # esperado: R0b -> R2 (indutivo puro)
diagnosticar(fp=0.3)                              # esperado: R0c (dados insuficientes p/ classificar)

# --- Caso sem regra aplicável
diagnosticar(fase=17)                             # falta o sentido da corrente

Fatos declarados: {'fase': 0}
[R1] Circuito puramente RESISTIVO (corrente em fase com a tensão, FP = 1).

Fatos declarados: {'fase': 90, 'corrente': 'atrasada'}
[R2] Circuito puramente INDUTIVO (corrente atrasada 90° em relação à tensão).

Fatos declarados: {'fase': 90, 'corrente': 'adiantada'}
[R3] Circuito puramente CAPACITIVO (corrente adiantada 90° em relação à tensão).

Fatos declarados: {'fase': 45, 'corrente': 'atrasada'}
[R4] Circuito RESISTIVO-INDUTIVO / RL (defasagem de 45°, corrente atrasada).

Fatos declarados: {'fase': 30, 'corrente': 'adiantada'}
[R5] Circuito RESISTIVO-CAPACITIVO / RC (defasagem de 30°, corrente adiantada).

Fatos declarados: {'xl': 50, 'xc': 30}
[R6] Predomínio INDUTIVO na malha (X_L=50Ω > X_C=30Ω).

Fatos declarados: {'xl': 20, 'xc': 80}
[R7] Predomínio CAPACITIVO na malha (X_C=80Ω > X_L=20Ω).

Fatos declarados: {'xl': 40, 'xc': 40}
[R8] Circuito em RESSONÂNCIA (X_L = X_C = 40Ω) — equivalente a RESISTIVO puro.

Fatos declarados: {'fp': 1}
[R0a] FP = 1 

[]